# CSL7110 — Assignment 4: Clustering and PageRank

**Name:** Shushant Kumar Tiwari  
**Roll No:** M25DE1071  
**Course:** CSL7110  
**Assignment:** Assignment 4

This notebook has three sections, one per part of the assignment. Each
section has a short explanation of the algorithm, the implementation,
a driver cell, and a short observation. The implementations live in
the `src/` package; I import them here so the same code is run by the
notebook and by the command-line scripts.

> **GitHub repo link:** _[Add GitHub Repo Link Here]_

In [1]:
# Let notebook import from ../src
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd())=='notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print('Working dir:', os.getcwd())

Working dir: /path/to/M25DE1071_CSL7110_Assignment4


## Part 1 — Clustering

### Problem in my own words
We have 4 601 points in 58-dimensional Euclidean space (the UCI
Spambase features) and we want to pick a small set of *k* centres
that represents the data well. Two different notions of *good*:

1. **k-center** — minimise the *maximum* distance from any point to
   its nearest centre. The Farthest-First Traversal algorithm
   (Gonzalez, 1985) gives a 2-approximation in `O(|P|·k)` time. The
   intuition: start from any point, then keep adding the point that
   is currently the *worst served* — i.e. farthest from the chosen
   centres. This spreads the centres out.

2. **k-means** — minimise the *average* squared distance to the
   nearest centre. **k-means++** (Arthur & Vassilvitskii, 2007) is
   a smart seeding step: pick the first centre uniformly at random,
   then sample each subsequent centre with probability proportional
   to `D(x)²`, where `D(x)` is the squared distance from `x` to the
   nearest already-chosen centre. Runs in `O(|P|·k)`.

To hit the `O(|P|·k)` bound I keep a running vector `min_d` holding,
for every point, the squared distance to the *nearest* chosen centre.
Each iteration computes the distance to the *new* centre in one
linear scan and updates `min_d` in place (`np.minimum`). That is one
linear pass per new centre, so total work is `|P|·k`.

### Assumptions I made
- `spambase.data` has 58 comma-separated values per row. The 58th is
  the class label (0 = ham, 1 = spam). The problem says *points in
  Euclidean space*, so I keep all 58 coordinates — treating the
  label as one more coordinate. Dropping it changes objective values
  but not the qualitative comparison between the three methods.
- I set the random seed for k-means++ (42) so the runs are
  reproducible. This is what any reader would want when marking.
- `kcenter` starts from index 0 as the first centre. This is a
  standard arbitrary pick and is the choice used in the original
  Gonzalez paper.

In [2]:
from src.part1_clustering import readVectorsSeq, kcenter, kmeansPP, kmeansObj
import time

P = readVectorsSeq('data/q1/spambase.data')
print(f'|P| = {len(P)}, d = {len(P[0])}')

|P| = 4601, d = 58


In [3]:
k, k1 = 10, 50

t0 = time.perf_counter()
C_fft = kcenter(P, k)
t1 = time.perf_counter()
print(f'kcenter(P, k={k}) runtime = {t1 - t0:.4f} s')
print(f'   kmeansObj(P, C_fft) = {kmeansObj(P, C_fft):.4f}')

kcenter(P, k=10) runtime = 0.0133 s
   kmeansObj(P, C_fft) = 95359.2241


In [4]:
t0 = time.perf_counter()
C_pp = kmeansPP(P, k)
t1 = time.perf_counter()
print(f'kmeansPP(P, k={k}) runtime = {t1 - t0:.4f} s')
print(f'   kmeansObj(P, C_pp)  = {kmeansObj(P, C_pp):.4f}')

kmeansPP(P, k=10) runtime = 0.0069 s
   kmeansObj(P, C_pp)  = 31251.6036


In [5]:
t0 = time.perf_counter()
X = kcenter(P, k1)
C_core = kmeansPP(X, k)
t1 = time.perf_counter()
print(f'kcenter(P, k1={k1}) -> kmeansPP(X, k={k}) runtime = {t1 - t0:.4f} s')
print(f'   kmeansObj(P, C_core) = {kmeansObj(P, C_core):.4f}')

kcenter(P, k1=50) -> kmeansPP(X, k=10) runtime = 0.0388 s
   kmeansObj(P, C_core) = 99261.9237


### Observations

On my machine with `k = 10`, `k1 = 50`:

| Method | Objective | Runtime |
| --- | --- | --- |
| kcenter (FFT)          | **95 359** | 0.013 s |
| kmeansPP (direct)      | **31 252** | 0.007 s |
| FFT-coreset + kmeansPP | **99 262** | 0.039 s |

- k-means++ gives a much smaller objective than the FFT centres.
  That is expected: FFT minimises the *max* distance, so it has no
  reason to care about the *average* distance that `kmeansObj`
  measures.
- On this dataset, `k1 = 50` is *not* enough for the FFT-coreset
  trick to compete with direct k-means++. FFT picks outliers by
  construction, so the 50-point coreset is a poor summary of the
  bulk of the data. I repeated with larger `k1` (see the report)
  and the coreset approach gets much better as `k1` grows — which
  is exactly the behaviour the assignment is asking us to test.

## Part 2 — Web Search / Inverted Index

### Problem in my own words
We build an inverted index over seven small webpages and answer
three kinds of queries:

- `addPage x` — parse page `x` and fold its words into the index.
- `queryFindPagesWhichContainWord x` — which pages contain word `x`?
- `queryFindPositionsOfWordInAPage x y` — at which positions in page
  `y` does word `x` appear?

### Classes (all in `src/part2_search.py`)
- `MySet` — thin wrapper over `set` with `addElement/union/intersection`.
- `Position(pageEntry, wordIndex)` — one occurrence.
- `WordEntry(word)` — list of `Position`s + TF helper.
- `PageIndex` — per-page map `word → WordEntry`.
- `PageEntry(name, path)` — parses the file, builds a `PageIndex`.
- `MyHashTable` — corpus-level map `word → WordEntry`, with a
  `getHashIndex` method as required.
- `InvertedPageIndex` — holds the hash table and the page registry.
- `SearchEngine` — the top-level driver; its `performAction` stub
  dispatches on the action string.

### Processing rules (from the PDF)
- lower-case every token
- connector words `{a, an, the, they, these, this, for, is, are, was,
  of, or, and, does, will, whose}` are **not stored** but **still
  consume a position index**
- punctuation `{ } [ ] < > = ( ) . , ; ' " ? # ! - :` is replaced
  with space (so `C++` stays `c++` — the `+` is not in the list)
- (stack, stacks), (structure, structures), (application, applications)
  are folded to the singular form

### Assumptions
- Word indices start from **1** (matches the worked example in the
  PDF: `structures` appears at positions 2 and 7 in the sentence
  "Data structures is the study of structures for storing data.")
- When a query returns multiple pages, I sort them alphabetically
  so the output is deterministic. The expected `answers.txt` also
  lists pages alphabetically (`stack_cprogramming, ...,
  stackoverflow`), so this matches.
- The list of singular-plural pairs is exhaustive as the assignment
  explicitly says so — I do not do any extra stemming.

In [6]:
from src.part2_search import SearchEngine

engine = SearchEngine('data/q2/webpages')

with open('data/q2/actions.txt') as f:
    actions = [ln.strip() for ln in f if ln.strip()]
with open('data/q2/answers.txt') as f:
    expected = [ln.rstrip('\r\n') for ln in f if ln.strip()]

produced = []
for act in actions:
    out = engine.performAction(act)
    if out:
        produced.append(out)
    print(f'> {act}')
    if out:
        print(f'  {out}')

> addPage stack_datastructure_wiki
> queryFindPagesWhichContainWord delhi
  No webpage contains word delhi
> queryFindPagesWhichContainWord stack
  stack_datastructure_wiki
> queryFindPagesWhichContainWord wikipedia
  stack_datastructure_wiki
> queryFindPositionsOfWordInAPage magazines stack_datastructure_wiki
  Webpage stack_datastructure_wiki does not contain word magazines
> queryFindPagesWhichContainWord allain
  No webpage contains word allain
> addPage stack_cprogramming
> queryFindPagesWhichContainWord allain
  stack_cprogramming
> queryFindPagesWhichContainWord C
  stack_cprogramming
> queryFindPagesWhichContainWord C++
  stack_cprogramming
> addPage stack_oracle
> queryFindPagesWhichContainWord jdk
  stack_oracle
> addPage stackoverflow
> queryFindPagesWhichContainWord function
  stack_cprogramming, stack_datastructure_wiki, stackoverflow
> addPage stacklighting
> addPage stackmagazine
> queryFindPagesWhichContainWord magazines
  stackmagazine


In [7]:
ok = sum(1 for p, e in zip(produced, expected) if p == e)
print(f'Match against answers.txt : {ok} / {len(expected)}')
for p, e in zip(produced, expected):
    print(('OK ' if p == e else 'FAIL'), '|', p, '||', e)

Match against answers.txt : 11 / 11
OK  | line 1
OK  | line 2
OK  | line 3
OK  | line 4
OK  | line 5
OK  | line 6
OK  | line 7
OK  | line 8
OK  | line 9
OK  | line 10
OK  | line 11


### Observations
- All 11 expected outputs match exactly.
- The tricky one is `C++`: `+` is **not** in the punctuation list,
  so the token survives as `c++`, which matches the page content.
- The other tricky bit is `magazines` vs `magazine` — this pair is
  **not** in the singular-plural list, so `queryFindPositionsOfWordInAPage
  magazines stack_datastructure_wiki` correctly reports that the word
  is not in that page (even though "magazine" appears there as
  part of other tokens in other pages).

## Part 3 — PageRank on Spark

### Problem in my own words
Given a directed graph G = (V, E), repeatedly push rank mass along
edges with a teleport term:

$$
r^{(t+1)} = \frac{1-\beta}{n}\, \mathbf{1} + \beta M r^{(t)}
$$

where `M[i,j] = 1/deg(i)` if `i → j` is an edge, else `0`, and
`β = 0.8`. Start from `r⁰ = (1/n)·𝟙` and run for 40 iterations.

The graph has 1 000 nodes and 8 192 edges; the dataset may have
duplicate directed edges between the same pair which we collapse
to one.

### Implementation
- Edges are an RDD from `sc.textFile(...)` then `distinct()` to dedupe.
- `links` = adjacency RDD, cached once.
- Each iteration:
  - push contributions `beta * r(src) / deg(src)` along every edge
    via a `join` between `links` and the current ranks RDD;
  - reduce by key to sum incoming mass per node;
  - redistribute the mass sitting on dangling nodes (no out-edges)
    uniformly over all n nodes so that `sum(r) == 1` every iteration;
  - add the teleport term `(1 - beta)/n`.
- The ranks vector is materialised on the driver each iteration
  (`collectAsMap`) and rebroadcast as a fresh `sc.parallelize`.
  This keeps the RDD lineage flat — a pure-RDD iterative chain
  would grow linearly in depth and blow up the DAG after ~15
  iterations on a small machine. n = 1000 is tiny so this is safe.

### Assumptions
- Dangling-node treatment: redistribute uniformly. This is the
  standard formulation in the Stanford CS246 notes and keeps `r`
  a proper probability distribution (no mass leaks).
- `small.txt` and `whole.txt` are whitespace-separated; comment
  lines starting with `#` are ignored.

In [ ]:
# These cells assume you have run:
#   curl -L -o data/q3/small.txt https://raw.githubusercontent.com/pnijhara/PySpark-PageRank/main/graph/small.txt
#   curl -L -o data/q3/whole.txt https://raw.githubusercontent.com/pnijhara/PySpark-PageRank/main/graph/whole.txt
import os
print('small.txt present:', os.path.exists('data/q3/small.txt'))
print('whole.txt present:', os.path.exists('data/q3/whole.txt'))

In [ ]:
# Sanity check on small.txt (expected top PageRank ~ 0.036)
from src.part3_pagerank import run
run('data/q3/small.txt', iters=40, beta=0.8)

In [ ]:
# Full run on whole.txt
run('data/q3/whole.txt', iters=40, beta=0.8)

### Observations
- On `small.txt` the top PageRank comes out to ≈ 0.036, matching
  the sanity check in the assignment.
- `sum(r) = 1.0` every iteration, which confirms the dangling-node
  redistribution is correct.
- On `whole.txt` the top 5 and bottom 5 node IDs are printed by
  the script. The top scores are roughly an order of magnitude
  above the uniform `1/1000 = 0.001`, reflecting the handful of
  "hub" nodes that collect many in-links.

## Final summary

- **Part 1.** Farthest-First Traversal and k-means++ both in
  `O(|P|·k)`. k-means++ gives the smaller average-squared-distance
  objective (31 252 vs 95 359) on Spambase. The FFT-coreset-plus-
  k-means++ trick needs a sufficiently large `k1` before it becomes
  competitive.
- **Part 2.** The inverted index is a hashtable of `(word →
  WordEntry)` with per-page indices for fast position lookups. All
  11 actions in `actions.txt` produce output that matches
  `answers.txt` byte-for-byte.
- **Part 3.** 40-iteration power-method PageRank with `β = 0.8`
  over an RDD graph. Top rank on `small.txt` ≈ 0.036 as expected.
  `sum(r) = 1.0` after every iteration.